In [1]:
## IMPORT LIBRARIES ----------------------------------------------------------------------------------------------------
#https://github.com/nnc-ufmg/circadipy/blob/main/src/circadipy/analysis_examples/intellicage/intellicage_analysis.ipynb
%matplotlib qt

%reload_ext autoreload
%autoreload 3

from ranking_methods import rank_accuracy
from circadipy import chrono_reader as chr  
import sys                                                                                                              # Import sys to add paths to libraries                                                                                                           # Import re to work with regular expressions
import glob                                                                                                             # Import glob to read files                                                                                                   # Import numpy to work with arrays and make calculations                                                                                            # Import time to measure time
import os                                                                                                               # Import path to work with paths                
import pandas as pd
from ranking_methods import build_all_proxies, evaluate_proxies, rank_accuracy
import numpy as np
from scipy import stats
from datetime import date, datetime, timedelta
from itertools import combinations
from scipy.stats import rankdata, spearmanr, kendalltau
import json
                                                                                      # Import pandas to work with dataframes
import warnings                                                                                                         # Import warnings to ignore warnings
warnings.filterwarnings('ignore')                                                                                       # Ignore warnings

## IMPORT CIRCADIPY ----------------------------------------------------------------------------------------------------

parent_path = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.append(parent_path)

## PCA Visualization - Dimensionality Reduction
from models import train_rf, train_gb, train_ridge, train_adaboost, train_extratrees, train_logistic
from summary import generate_summary_report
from util import  get_data_scaled, generate_features, generate_temporal_features, calculate_correlations, build_animal_protocols, get_sorted_animals_files, combine_all_features
from analysis_visualization import generate_methods_comparison, plot_animals_activity, plot_correlation, plot_pca_analysis, plot_cross_correlation, summary_visualization

In [11]:
best_feat = 'ibi_median'
#data_folder = "./data/dados_2026_06_08"
data_folder = "./data/dados_2026_05_01"
#data_folder = "./data/dados_iniciais_estruturados"
dataFiles = glob.glob(f'{data_folder}/*.zip')
output_folder = './results/predict_to_predict'
ground_file = f'{data_folder}/ground.json'
os.makedirs(output_folder, exist_ok=True)
print(dataFiles)

if os.path.exists(ground_file):
    with open(ground_file, 'r') as f:
        ranks_ground = json.load(f)
        #ranks_ground = [int(f.split("_")[1]) for f in list(ranks_ground.keys())]

for k, v in ranks_ground.items():
    print(f"{k}: {v}")
root_folder = f"{data_folder}"    



['./data/dados_2026_05_01/2026-04-16 17.41.32.zip', './data/dados_2026_05_01/2026-04-23 17.58.52.zip', './data/dados_2026_05_01/2026-04-14 17.19.07.zip', './data/dados_2026_05_01/2026-04-24 17.31.38.zip', './data/dados_2026_05_01/2026-04-22 10.43.23.zip', './data/dados_2026_05_01/2026-04-24 19.33.09.zip', './data/dados_2026_05_01/2026-04-17 19.18.35.zip']
animal_6: 1
animal_10: 2
animal_11: 3
animal_9: 4
animal_1: 5
animal_7: 6
animal_2: 7
animal_12: 8
animal_4: 9
animal_8: 10
animal_5: 11
animal_3: 12


In [12]:

for file in dataFiles:
    zip_folder = file.split("/")[-1]
    sub_folder = zip_folder.split(".")[0].replace(" ", "_")
    os.makedirs(os.path.join(root_folder, sub_folder), exist_ok=True)
    a = chr.intellicage_unwrapper([file], sub_folder, sampling_interval = '30T')



File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_1.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_10.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_11.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_12.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_2.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_3.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_4.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_5.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_6.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_7.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_8.txt
File saved in ./data/dados_2026_05_01/2026-04-16_17/animal_9.txt
File saved in ./data/dados_2026_05_01/2026-04-23_17/animal_1.txt
File saved in ./data/dados_2026_05_01/2026-04-23_17/animal_10.txt
File saved in ./data/dados_2026_05_01/2026-04-23_17/animal_11.txt
File saved in ./data

In [13]:
individual_files = glob.glob(root_folder + "/**/*.txt", recursive=True)
individual_files = [f for f in individual_files if "animal_" in f]

start_date = date(2026, 3, 17)
end_date   = date(2026, 3, 20)
dates_to_keep = [(start_date + timedelta(days=i)).strftime("%Y-%m-%d")
                for i in range((end_date - start_date).days + 1)]


if dates_to_keep:
    individual_files = [f for f in individual_files if f.split("/")[-2].split("_")[0] in dates_to_keep]

apply_filtering = True
animals = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
animals_files = get_sorted_animals_files(individual_files, animals)
animals_protocols, animals_by_day = build_animal_protocols(animals_files, apply_filtering=apply_filtering)

In [ ]:
output_path = f'{output_folder}/basic_features.csv'
features_df = generate_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/temporal_features.csv'
temporal_df = generate_temporal_features(animals_protocols, output_path=output_path)

output_path = f'{output_folder}/all_features.csv'
all_features, feature_cols = combine_all_features(features_df, temporal_df, output_path=output_path)

X_scaled =  get_data_scaled(all_features, feature_cols)
y = []
for animal in all_features['animal'].tolist():
    y.append(ranks_ground[animal])

print(y)
# [9, 8, 11, 10, 12, 1, 5, 6, 7, 2, 3, 4]

all_features.head()

Saving features on ./results/predict_to_predict/basic_features.csv
Saving temporal features on ./results/predict_to_predict/temporal_features.csv
Saving all features on ./results/predict_to_predict/all_features.csv


ValueError: Found array with 0 sample(s) (shape=(0, 0)) while a minimum of 1 is required by StandardScaler.

In [6]:
feature_rhos_path = "./data/dados_iniciais_estruturados/feature_rhos.csv"
feature_rhos_df = pd.read_csv(feature_rhos_path)

if {'feature', 'rho'}.issubset(feature_rhos_df.columns):
    feature_rhos = feature_rhos_df.set_index('feature')['rho']
else:
    raise ValueError(
        "`feature_rhos_previous.csv` must contain `feature` and `rho` columns. "
        "Recreate it with: feature_rhos.to_frame('rho').rename_axis('feature').to_csv(...)"
    )

feature_rhos.head()

feature
rhythm_ratio_24_over_harm    0.790210
power_24h                    0.706294
autocorr_12h                -0.664336
power_12h                   -0.657343
cosinor_amplitude            0.657343
Name: rho, dtype: float64

In [7]:


best_feat = 'cosinor_amplitude'
combos = [['cosinor_amplitude'], ['power_24h', 'high_activity_frac', 'activity_per_bout'], ['power_24h', 'short_gap_frac', 'night_ibi_cv']]



result = {}
cont = 0
for named_combo in combos:

    feature_rhos, proxies_raw = build_all_proxies(
        all_features, feature_cols, X_scaled, None,
        k=3, best_feat_idx=best_feat,
        named_combo=named_combo,
        feature_rhos=feature_rhos
    )
    for k, v in proxies_raw.items():
        print(f"{k}: {v}")



    best_feature_key = f'Best feature ({best_feat})'

    scores = proxies_raw[best_feature_key]
    pred_rank = rankdata(scores, method='ordinal')
    #print(pred_rank)


    output_best_feature = f'{data_folder}/pred_{best_feat}.csv'

    best_combo_key = f'Best combo ({ " + ".join(named_combo) })'
    scores = proxies_raw[best_combo_key]
    pred_rank = rankdata(scores, method='ordinal')


    combo_name = "_".join(named_combo)

    if combo_name not in result:
        result[combo_name] = {"pred": pred_rank, "ground": y}


# df = pd.DataFrame(result)
# df.to_csv(f"{root_folder}/pred_data_to_predict.csv", index=False)


print(result)


Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.24117553 0.2809962  0.47956141 0.44404151 0.42746586 0.38022122
 0.26038611 0.40175124 0.3380771  0.29245193 0.37392806 0.26916381]
Best combo (cosinor_amplitude): [-1.40897792 -0.88911819  1.70315471  1.23944159  1.02304615  0.40626647
 -1.15818333  0.68734123 -0.14392596 -0.7395635   0.32410904 -1.04359028]
Combo sign-aligned mean (cosinor_amplitude): [-1.40897792 -0.88911819  1.70315471  1.23944159  1.02304615  0.40626647
 -1.15818333  0.68734123 -0.14392596 -0.7395635   0.32410904 -1.04359028]
Using feature_rhos provided externally (e.g. from a reference dataset).
Best feature (cosinor_amplitude): [0.24117553 0.2809962  0.47956141 0.44404151 0.42746586 0.38022122
 0.26038611 0.40175124 0.3380771  0.29245193 0.37392806 0.26916381]
Best combo (power_24h + high_activity_frac + activity_per_bout): [-2.60449622  1.5078055   2.93185325  0.54446378 -0.17062289 -0.17918256
 -1.54022

In [8]:
for combo_name, r in result.items():
    print(combo_name)
    print(r['pred'])
    print(y)
    print(rank_accuracy(r['pred'], y))
    print()


cosinor_amplitude
[ 1  4 12 11 10  8  2  9  6  5  7  3]
[5, 7, 12, 9, 11, 1, 6, 10, 4, 2, 3, 8]
{'accuracy': 0.08333333333333333, 'within_1': 0.25, 'within_2': 0.4166666666666667, 'mae': 3.0, 'rho': 0.4755244755244756}

power_24h_high_activity_frac_activity_per_bout
[ 1 10 12  9  7  6  2 11  4  8  5  3]
[5, 7, 12, 9, 11, 1, 6, 10, 4, 2, 3, 8]
{'accuracy': 0.25, 'within_1': 0.3333333333333333, 'within_2': 0.4166666666666667, 'mae': 2.8333333333333335, 'rho': 0.4825174825174825}

power_24h_short_gap_frac_night_ibi_cv
[ 2  1 12  5 11  8  6  9 10  3  7  4]
[5, 7, 12, 9, 11, 1, 6, 10, 4, 2, 3, 8]
{'accuracy': 0.25, 'within_1': 0.4166666666666667, 'within_2': 0.4166666666666667, 'mae': 3.0, 'rho': 0.3706293706293707}

